# Explore Frontier Game
Select the project `.venv` kernel after installing `.[dev]`. Restart and Run All.
The package provides the simulation; this notebook asks questions. All units are
arbitrary and policies are fixed, not learned or equilibrium solutions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from frontier_game import Config, FixedPolicy, simulate, run_trials, summarize

config = Config()


## 1. Where does safety fall behind capability?

In [ ]:
episode = simulate(config, FixedPolicy(0.8), FixedPolicy(0.4),
                   np.random.default_rng(42), trace=True)
trajectory = pd.DataFrame(episode["history"])
trajectory.plot(x="step", y=["capability_a", "capability_b", "safety"])
plt.ylabel("Arbitrary model units")
plt.title("One sampled episode (may end in catastrophe)")
plt.show()
trajectory.tail()


## 2. Does unilateral racing pay?
Independent seeds distinguish each policy pair. Read both payoff columns and intervals.

In [ ]:
comparisons = []
for index, (a, b) in enumerate([(0.4, 0.4), (0.4, 0.8), (0.8, 0.4), (0.8, 0.8)]):
    sample = run_trials(config, FixedPolicy(a), FixedPolicy(b), trials=1000, seed=100+index)
    table = summarize(sample)
    table["a"], table["b"] = a, b
    comparisons.append(table)
comparison = pd.concat(comparisons, ignore_index=True)
comparison


In [ ]:
comparison.pivot(index="a", columns=["metric", "b"], values="mean")[["payoff_a", "payoff_b"]]


## 3. How does sampling uncertainty change with N?
Nested prefixes share episodes; these points are correlated. Error bars estimate
sampling uncertainty, not distance to a known true value. Compare widths with 1/sqrt(N).


In [ ]:
sample = run_trials(config, FixedPolicy(0.8), FixedPolicy(0.4), trials=4000, seed=2026)
convergence = pd.DataFrame([
    summarize(sample.iloc[:n]).set_index("metric").loc["payoff_a"].to_dict()
    for n in [100, 250, 1000, 4000]
])
plt.errorbar(convergence["n"], convergence["mean"],
             yerr=convergence["ci_high"]-convergence["mean"], fmt="o-", capsize=4)
plt.xscale("log")
plt.xlabel("Episodes")
plt.ylabel("A mean payoff, approximate 95% interval")
plt.show()
convergence[["n", "mean", "standard_error"]]


## Your interpretation
1. Hold B fixed: does A gain by moving from 0.4 to 0.8? How uncertain is that?
2. Change `catastrophe_cost` in Config and rerun all cells. What changes and why?
3. Which conclusion depends on shared safety or the terminal winner prize?
4. Why does this table not prove a dynamic-game equilibrium?

Write your answers here before expanding the model.
